# Suite PYPRS — Parsing Python

The versions of Python parse source text into Python trees by delegating to tree-sitter-python, then rewriting its
concrete syntax tree into Python kinds. The converter regroups what tree-sitter-python 0.25 groups unlike Python, and
a pre-pass handles the constructs of Python 3.13 and later that it cannot parse. Each case parses constructs and
checks the tree (by shape where it matters, and by the text it prints), then the errors, each with its line and
column. In Python, CPython's own parser checks the conformance source too.

In [ ]:
import ast
from pathlib import Path

from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Errors import ParseError
from mbse.Programs.Python import Python312, Python314, Syntax as S, _Parser
from mbse.Programs.Python._Standard import PythonStandard
from support import raises

PY315 = PythonStandard(3, 15)
SOURCES = Path("../../conformance/sources")


def parse(text, standard=PY315):
    module = standard.parse(text)
    assert S.LANGUAGE.validate(module) == [], S.LANGUAGE.validate(module)
    return module


def first(text, standard=PY315):
    return parse(text, standard).body[0]


def value(text):
    """The expression of the expression statement `text`."""
    return first(text).value


def same_text(text, expected=None, standard=PY315):
    """Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree."""
    module = parse(text, standard)
    printed = standard.print(module)
    assert printed == (text if expected is None else expected) + "\n", printed
    assert F.same(parse(printed, standard), module)
    return module


def fails(text, message, line, column, standard=PY315):
    with raises(ParseError, match=message):
        standard.parse(text)
    try:
        standard.parse(text)
    except ParseError as error:
        assert (error.line, error.column) == (line, column), (error.line, error.column)


def kinds(nodes):
    return [type(n).__name__ for n in nodes]

## PYPRS-01 · The conformance source parses to a valid tree, which prints and parses back the same, and means what CPython reads

In [ ]:
text = (SOURCES / "python314.py").read_text(encoding="utf-8")
module = parse(text, Python314)
printed = Python314.print(module)
assert F.same(parse(printed, Python314), module) and Python314.print(parse(printed, Python314)) == printed
assert ast.dump(ast.parse(printed)) == ast.dump(ast.parse(text))  # what CPython itself reads is the same
assert {type(n).__name__ for n in F.walk(module)} == set(S.LANGUAGE.kinds())
parsed, positions, _ = _Parser.parse(text)
assert all(id(n) in positions for n in F.walk(parsed))  # every node is placed, for the version's problems
print(len(printed.splitlines()), "lines")

## PYPRS-02 · Literals keep their spelling: numbers, strings and bytes, prefixes and quotes, and concatenation

In [ ]:
for literal in ("0", "1_000", "0x_FF", "0o17", "0b1010", "1.5e-3", "2j", "3.0J", ".5", "5.", "None", "True", "False",
                "...", "'a'", '"a"', "'''a'''", '"""a\nb"""', "b'a'", "rb'\\d'", "R'a'", "u'a'", "'\\n\\u00e9'"):
    assert F.same(value(literal), S.Constant(spelling=literal))
    same_text(literal)
concatenated = value("'a' \"b\" f'{c}'")
assert kinds(concatenated.values) == ["Constant", "Constant", "JoinedStr"]
same_text("('a'\n 'b')", "('a' 'b')")  # layout is the printer's
print("ok")

## PYPRS-03 · f-strings and t-strings: their text, replacement fields, conversions, format specs and self-documenting fields

In [ ]:
joined = value("f'a{b!r:>{w}} {c = } {{d}}'")
assert (joined.prefix, joined.quote) == ("f", "'")
assert kinds(joined.values) == ["StringText", "FormattedValue", "StringText", "FormattedValue", "StringText"]
field = joined.values[1]
assert (field.text, field.conversion, field.debug) == ("b", "r", False)
assert kinds(field.format_spec.values) == ["StringText", "FormattedValue"]
assert field.format_spec.values[0].spelling == ">" and field.format_spec.values[1].text == "w"
assert (joined.values[3].text, joined.values[3].debug) == ("c = ", True)
assert joined.values[4].spelling == " {{d}}"
template = value("t'{x!s:{y}}'")
assert isinstance(template, S.TemplateStr) and isinstance(template.values[0], S.Interpolation)
assert isinstance(template.values[0].format_spec.values[0], S.FormattedValue)
assert value("rf'\\\\'").values[0].spelling == "\\\\"  # which tree-sitter-python counts as part of the quote
assert value('f"{x:=10}"').values[0].format_spec.values[0].spelling == "=10"  # a spec, not an assignment
assert isinstance(value('f"{(x:=10)}"').values[0].value, S.Parenthesized)
assert value('F"""\n{x}\n"""').quote == '"""' and value("Rf''").values == []
for text in ("f'a{b!r:>{w}} {c = } {{d}}'", "t'{x!s:{y}}'", "rf'\\\\'", 'f"{x:=10}"', 'f"{x = !r:^10}"',
             "f'{\"nested\"}'", 'f"{f\'{x}\'}"', "f'{x,y}'", "f'{a, *b}'", "f'{yield x}'"):
    same_text(text)
print("ok")

## PYPRS-04 · Expressions, with Python's precedence where tree-sitter-python's differs

In [ ]:
regrouped = value("~a ^ b & c | d & e ^ -f")  # `&` binds tighter than `^`
assert Python314.print(regrouped) == "~a ^ b & c | d & e ^ -f"
assert (regrouped.op, regrouped.left.op, regrouped.left.right.op) == ("|", "^", "&")
power = value("a ** b ** c")
assert power.right.op == "**"  # `**` groups to the right
assert value("a - b - c").left.op == "-"
awaited = value("await x ** 2")
assert isinstance(awaited, S.BinOp) and isinstance(awaited.left, S.Await)  # `(await x) ** 2`
walrus = value("(m := a if b else c)").value
assert isinstance(walrus, S.NamedExpr) and isinstance(walrus.value, S.IfExp)
assert kinds(value("a < b <= c is not d not in e").comparisons) == ["Comparison"] * 4
assert [c.op for c in value("a is not b not in c").comparisons] == ["is not", "not in"]
assert [c.op for c in value("a is  not b").comparisons] == ["is not"]
assert len(value("a and b and c").values) == 3 and len(value("a or b and c").values) == 2
assert len(value("a and (b and c)").values) == 2
assert isinstance(value("not a"), S.UnaryOp) and value("-a").op == "-"
call = value("f(a, *b, c=1, **d)")
assert kinds(call.args) == ["Name", "Starred"] and call.keywords[1].arg is None
assert kinds(value("f(x for x in y)").args) == ["GeneratorExp"]
assert kinds(value("a[1:2, ::3, *s]").slice.elts) == ["Slice", "Slice", "Starred"]
assert value("a[1,]").slice.elts[0].spelling == "1" and isinstance(value("a[1]").slice, S.Constant)
assert F.same(value("a[::]").slice, S.Slice())
assert isinstance(value("()"), S.Tuple) and isinstance(value("(1,)"), S.Parenthesized)
assert kinds(value("1, 2").elts) == ["Constant", "Constant"] and kinds(value("f(),").elts) == ["Call"]
assert value("{**a, 'b': 1}").items[0].key is None
for text in ("a + b * c", "(a + b) * c", "-a ** b", "(-a) ** b", "a if b else c if d else e", "lambda: 0",
             "lambda a, /, b=1, *c, d, **e: a", "[x for x in y if x if not x]", "{k: v for k, v in d}",
             "{x for x in y}", "(x for x in y)", "[x async for x in y]", "[*a, *b]", "{1, *a}", "{**a, 'b': 1}",
             "a.b.c", "a[b][c]", "f(a)(b)", "await a", "yield", "yield a, b", "x = yield", "(yield from a)",
             "a[b:c]", "a[:]", "a[b:]", "a[:c]", "a[::d]", "f(*(a for a in b))", "print >> f, a", "print >> f"):
    same_text(text)
same_text("f((x for x in y))", "f(x for x in y)")  # the call's parentheses are the generator's
python2 = value("[x for x in 1, 2]")  # Python 2's iterable, which tree-sitter-python reads, prints as Python 3's
assert isinstance(python2.generators[0].iter, S.Tuple) and PY315.print(python2) == "[x for x in (1, 2)]"
print("ok")

## PYPRS-05 · Assignment targets, and annotated and augmented assignment

In [ ]:
assign = first("a = b = c")
assert kinds(assign.targets) == ["Name", "Name"]
assert kinds(first("a, *b = c").targets[0].elts) == ["Name", "Starred"]
assert isinstance(first("[a, b] = c").targets[0], S.List)
assert isinstance(first("(a, b) = c").targets[0].value, S.Tuple) and isinstance(first("(a) = c").targets[0],
                                                                                  S.Parenthesized)
assert F.same(first("() = c").targets[0], S.Tuple())
assert first("x += 1").op == "+" and first("x //= 1").op == "//" and first("x **= 1").op == "**"
annotated = first("x: int = 1")
assert isinstance(annotated, S.AnnAssign) and annotated.value is not None and first("x: int").value is None
assert isinstance(first("a.b: int").target, S.Attribute)
renamed = first("type(t).__name__ = 'x'")  # an assignment, which tree-sitter-python reads as a type alias
assert isinstance(renamed, S.Assign) and renamed.targets[0].value.func.id.spelling == "type"
union = first("def f(a: list[str] | str | None): pass").args.args[0].annotation
assert union.left.op == "|" and isinstance(union.left.left, S.Subscript)  # `|` groups to the left
for text in ("a = b = c", "a, *b = c", "[a, b] = c", "(a, b) = c", "(a,) = c", "() = c", "a.b[c] = d", "x @= 1",
             "x: list[int] | None = None", "x: int", "a = yield b", "for (a, b), c in d:\n    pass",
             "type(t).__name__ = 'x'", "type(m)[0] = p", "type[0] = 1"):
    same_text(text)
print("ok")

## PYPRS-06 · Simple statements: imports, global and nonlocal, del, assert, raise, return and type aliases

In [ ]:
imported = first("import a.b.c as d, e")
assert [a.name.names[-1].spelling for a in imported.names] == ["c", "e"] and imported.names[0].asname.spelling == "d"
relative = first("from ..m import x as y, z")
assert (relative.level, relative.module.names[0].spelling, len(relative.names)) == (2, "m", 2)
assert first("from . import x").module is None and first("from . import x").level == 1
assert first("from m import *").names[0].name is None and first("from m import x").level is None
future = first("from __future__ import annotations")
assert future.module.names[0].spelling == "__future__" and future.names[0].name.names[0].spelling == "annotations"
assert first("from ... import x").level == 3
alias = first("type A[T] = list[T]")
assert isinstance(alias, S.TypeAlias) and alias.name.id.spelling == "A" and isinstance(alias.value, S.Subscript)
for text in ("import a.b.c as d, e", "from ..m import x as y, z", "from . import x", "from m import *",
             "from __future__ import annotations as an", "global a, b", "def f():\n    nonlocal a",
             "del a, b[0], c.d", "assert a", "assert a, 'm'", "raise", "raise E", "raise E from c", "return",
             "def f():\n    return a, b", "type A = int", "type A[T: int, *Ts, **P] = Callable[P, tuple[T, *Ts]]",
             "type A = B | C | D", "pass", "while a:\n    break", "while a:\n    continue"):
    same_text(text)
same_text("from m import (a,\n    b)", "from m import a, b")
print("ok")

## PYPRS-07 · Compound statements: conditions, loops, exceptions, context managers and matches

In [ ]:
chain = first("if a:\n    pass\nelif b:\n    pass\nelif c:\n    pass\nelse:\n    pass")
assert isinstance(chain.orelse[0], S.If) and isinstance(chain.orelse[0].orelse[0], S.If)
assert kinds(chain.orelse[0].orelse[0].orelse) == ["Pass"]
assert isinstance(first("async def f():\n    async for a in b:\n        pass").body[0], S.AsyncFor)
tried = first("try:\n    pass\nexcept (A, B) as e:\n    pass\nexcept C:\n    pass\nexcept:\n    pass\n"
              "else:\n    pass\nfinally:\n    pass")
assert [h.name.spelling if h.name else None for h in tried.handlers] == ["e", None, None]
assert isinstance(tried.handlers[0].type, S.Parenthesized) and tried.handlers[2].type is None
assert isinstance(first("try:\n    pass\nexcept A, B:\n    pass").handlers[0].type, S.Tuple)
assert isinstance(first("try:\n    pass\nexcept* A:\n    pass"), S.TryStar)
with_ = first("with a as b, c:\n    pass")
assert with_.items[0].optional_vars.id.spelling == "b" and with_.items[1].optional_vars is None
assert first("with (a as b):\n    pass").items[0].optional_vars.id.spelling == "b"
assert isinstance(first("async def f():\n    async with a:\n        pass").body[0], S.AsyncWith)
assert isinstance(first("match a, b:\n    case _:\n        pass").subject, S.Tuple)
assert isinstance(first("match a,:\n    case _:\n        pass").subject, S.Tuple)
for text in ("if a:\n    pass\nelif b:\n    pass\nelse:\n    pass", "while a:\n    pass\nelse:\n    pass",
             "for a in b:\n    pass\nelse:\n    pass", "try:\n    pass\nfinally:\n    pass",
             "try:\n    pass\nexcept* (A, B) as e:\n    pass", "try:\n    pass\nexcept A, B:\n    pass",
             "with a as (b, c), d:\n    pass", "match a:\n    case 1:\n        pass\n    case _:\n        pass",
             "if a:\n    if b:\n        pass\nelse:\n    pass"):
    same_text(text)
same_text("with (open(a) as b, c):\n    pass", "with open(a) as b, c:\n    pass")
same_text("if a: b = 1; c = 2", "if a:\n    b = 1\n    c = 2")
print("ok")

## PYPRS-08 · Definitions: functions, classes, their decorators, parameters and type parameters

In [ ]:
function = first("@d\n@e.f(1)\nasync def f[T](a, /, b: int = 1, *c: *Ts, d, e=2, **f: int) -> T:\n    pass")
assert isinstance(function, S.AsyncFunctionDef) and kinds(function.decorator_list) == ["Name", "Call"]
args = function.args
assert [a.arg.spelling for a in args.posonlyargs] == ["a"] and [a.arg.spelling for a in args.args] == ["b"]
assert args.vararg.arg.spelling == "c" and isinstance(args.vararg.annotation, S.Starred)
assert [a.arg.spelling for a in args.kwonlyargs] == ["d", "e"] and args.kwonlyargs[1].default_value is not None
assert args.kwarg.annotation.id.spelling == "int" and function.returns.id.spelling == "T"
assert [a.arg.spelling for a in first("def f(*, a): pass").args.kwonlyargs] == ["a"]
generic = first("class C[T: int, U: (int, str), *Ts, **P](B, metaclass=M, **k):\n    pass")
assert kinds(generic.type_params) == ["TypeVar", "TypeVar", "TypeVarTuple", "ParamSpec"]
assert isinstance(generic.type_params[1].bound, S.Parenthesized)
assert kinds(generic.bases) == ["Name"] and [k.arg.spelling if k.arg else None for k in generic.keywords] == ["metaclass",
                                                                                                         None]
assert first("class C:\n    pass").bases == []
for text in ("@d\n@e.f(1)\nasync def f[T](a, /, b: int = 1, *c: *Ts, d, e=2, **f: int) -> T:\n    pass",
             "def f(*, a):\n    pass", "def f(a, /):\n    pass", "def f(*a, **b):\n    pass", "@(lambda f: f)\ndef f():\n    pass",
             "class C[T: int, U: (int, str), *Ts, **P](B, metaclass=M, **k):\n    pass", "class C:\n    pass",
             "class C(B):\n    x: int = 0"):
    same_text(text)
same_text("class C():\n    pass", "class C:\n    pass")
print("ok")

## PYPRS-09 · Patterns: values, singletons, sequences, mappings, classes, captures, alternatives and groups

In [ ]:
def case(pattern):
    return first(f"match x:\n    case {pattern}:\n        pass").cases[0].pattern


assert isinstance(case("1"), S.MatchValue) and case("-1").value.op == "-" and case("-1 + 2j").value.op == "+"
assert isinstance(case("a.b"), S.MatchValue) and case("a").name.spelling == "a"
assert case("_").name is None and case("_").pattern is None
assert [case(s).value for s in ("None", "True", "False")] == ["None", "True", "False"]
assert [case(s).delimiters for s in ("[a]", "(a,)", "a, b", "()")] == ["[]", "()", None, "()"]
assert isinstance(case("(a)"), S.MatchAs)  # a group is its pattern
group = case("1 | (2 | 3)")
assert kinds(group.patterns) == ["MatchValue", "MatchOr"]
assert len(case("1 | 2 | 3").patterns) == 3 and kinds(case("-1 | _").patterns) == ["MatchValue", "MatchAs"]
mapping = case("{'k': v, -1: _, **rest}")
assert kinds(mapping.keys) == ["Constant", "UnaryOp"] and mapping.rest.spelling == "rest"
assert kinds(case("{a.b: x, 1 + 2j: y}").keys) == ["Attribute", "BinOp"]
cls = case("P(1, x=-2, y=_, z=a.b, w=str() as s)")
assert [a.spelling for a in cls.kwd_attrs] == ["x", "y", "z", "w"]
assert kinds(cls.kwd_patterns) == ["MatchValue", "MatchAs", "MatchValue", "MatchAs"]
assert cls.kwd_patterns[3].name.spelling == "s" and isinstance(cls.kwd_patterns[3].pattern, S.MatchClass)
assert case("[*a, *_]").patterns[1].name is None and case("(1 | 2) as s").name.spelling == "s"
for pattern in ("1", "-1", "-1 + 2j", "1.5 - 2j", "'a' 'b'", "b'a'", "a.b.c", "a", "_", "None", "[a, *b]", "(a,)",
                "()", "[]", "{'k': v, -1: _, **rest}", "{}", "{a.b: x, -1 - 2j: y}", "P(1, x=-2, y=_, z=a.b, w=str() as s)", "P()", "1 | 2 | 3",
                "1 | (2 | 3)", "1 | 2 as s", "[a, b] as c", "[(a as b) as c]", "a, b", "a,", "[a | b]"):
    same_text(f"match x:\n    case {pattern}:\n        pass")
same_text("match x:\n    case (a):\n        pass", "match x:\n    case a:\n        pass")
same_text("match x:\n    case (1 | 2) as s:\n        pass", "match x:\n    case 1 | 2 as s:\n        pass")
print("ok")

## PYPRS-10 · Comments are kept where statements are listed, on their own lines or trailing; elsewhere they are dropped

In [ ]:
module = parse("# lead\nx = 1  # trailing\nif a:  # header\n    pass\n    # end of block\n# between\nelif b:\n    pass\n"
               "else:\n    # first\n    pass\ny = (1,  # dropped\n     2)\n")
comments = [(c.text, c.trailing) for c in F.walk(module) if isinstance(c, S.Comment)]
assert comments == [(" lead", False), (" trailing", True), (" header", True), (" end of block", False),
                    (" between", False), (" first", False)]
assert kinds(module.body[3].body) == ["Comment", "Pass", "Comment", "Comment"]
match = first("match a:\n    case 1:\n        pass\n    # between cases\n    case 2:\n        pass\n")
assert kinds(match.cases[0].body) == ["Pass", "Comment"]
tried = first("try:\n    pass\n# before except\nexcept E:\n    pass\n# before finally\nfinally:\n    pass\n")
assert kinds(tried.body) == ["Pass", "Comment"] and kinds(tried.handlers[0].body) == ["Pass", "Comment"]
for text in ("# lead\nx = 1  # trailing", "if a:  # header\n    pass\nelse:  # else\n    pass",
             "def f():  # header\n    pass  # trailing\n    # own"):
    same_text(text)
same_text("x = 1 \\\n    + 2", "x = 1 + 2")  # line continuations are layout
print("ok")

## PYPRS-11 · Python 3.13 and later, which the pre-pass puts back: lazy imports, dict comprehensions that unpack, and type parameters' defaults

In [ ]:
lazy = parse("lazy import json\nlazy from . import x\nimport os\n")
assert [s.is_lazy for s in lazy.body] == [True, True, False]
assert isinstance(value("{**d for d in ds}"), S.DictComp) and value("{**d for d in ds}").value is None
assert isinstance(value("{**a, **b}"), S.Dict)  # a display, not a comprehension
assert isinstance(value("{**a for a in b}\n[{**c} for c in d]"), S.DictComp)
defaults = first("def f[T = int, *Ts = (int,), **P = [str], U: str = 'u'](): pass").type_params
assert [type(d.default_value).__name__ for d in defaults] == ["Name", "Parenthesized", "List", "Constant"]
assert defaults[3].bound.id.spelling == "str"
assert first("class C[T = dict[str,\n    int]]: pass").type_params[0].default_value.slice.elts[1].id.spelling == "int"
assert first("type A[T = int] = list[T]").type_params[0].default_value.id.spelling == "int"
assert parse("lazy = 1\nlazy(import_)\n").body[0].targets[0].id.spelling == "lazy"  # still a name
assert isinstance(parse("lazy import x\ny = {**a, 'b': 1}\n").body[1].value, S.Dict)
for text in ("lazy import json", "lazy from . import x", "{**d for d in ds}", "{**a, **b}",
             "def f[T = int, *Ts = (int,), **P = [str]]():\n    pass", "type A[T = int] = list[T]",
             "class C[T: str = 'u']:\n    pass"):
    same_text(text)
print("ok")

## PYPRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("x = (", "syntax error", 1, 1)  # where tree-sitter's error starts
fails("é = (1,\n     é é)", "syntax error", 2, 8)
fails("print 'x'", "unsupported syntax: print_statement, which is Python 2", 1, 1)
fails("exec 'x'", "unsupported syntax: exec_statement, which is Python 2", 1, 1)
fails("a <> b", "unsupported syntax: <>, which is Python 2", 1, 3)
fails("def f((a, b)): pass", "unsupported syntax: tuple_pattern", 1, 7)
fails("def f((a, b)=1): pass", "unsupported syntax: tuple_pattern", 1, 7)
fails("x = a[*b, *(c)]", "syntax error", 1, 11)  # tree-sitter-python 0.25 cannot parse it
fails("def f[T = int](): pass", "TypeVar.default_value needs Python 3.13, but this is Python 3.12", 1, 7, Python312)
fails("\n\nx = t'{y}'", "TemplateStr needs Python 3.14, but this is Python 3.12", 3, 5, Python312)
fails("lazy import a", "Import.is_lazy needs Python 3.15, but this is Python 3.14", 1, 1, Python314)
fails("[*a for a in b]", "ListComp of unpacked elements needs Python 3.15, but this is Python 3.14", 1, 1, Python314)
fails("lazy import (", "syntax error", 1, 1)
fails("def f(:", "expected )", 1, 7)
fails("def f[a.b](): pass", "unsupported syntax: attribute", 1, 7)
fails("x = (a as b)", "unsupported syntax: as_pattern", 1, 6)
fails("def f[T = ](): pass", "syntax error", 1, 9)
fails("def f[T = (](): pass", "syntax error", 1, 9)
fails("def f[\n  T = 1 +](): pass", "syntax error", 2, 9)  # in a default, which is parsed on its own
fails("\U0001d465 = (1,\n     \U0001d465 \U0001d465)", "syntax error", 2, 8)  # beyond UTF-16's first plane, one each
print("ok")